# VGG19 + EHOA with Novelty: Hybrid Training, Adaptive Population, and Monte Carlo Dropout
This notebook implements a complete VGG19-based pipeline with EHOA hyperparameter optimization, adaptive population control, momentum-driven search updates, gradient-based fine-tuning, and uncertainty-aware fraud risk scoring using Monte Carlo Dropout.

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras import backend as K

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [2]:
CSV_PATH = "archive/data.csv"
IMAGE_DIR = "archive/image"

assert os.path.exists(CSV_PATH), f"CSV not found: {CSV_PATH}"
df = pd.read_csv(CSV_PATH)
assert "classes" in df.columns, "Expected 'classes' column in CSV"

image_column = df.columns[0]

df["image_path"] = df[image_column].astype(str).apply(
    lambda x: os.path.join(IMAGE_DIR, x + ".jpeg")
)

df = df.dropna(subset=["image_path", "classes"])
df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
print("Total valid samples:", len(df))
print("Class distribution:\n", df["classes"].value_counts())

Total valid samples: 1512
Class distribution:
 classes
unknown           549
door_dent         192
door_scratch      154
glass_shatter     137
tail_lamp         136
head_lamp         133
bumper_dent       129
bumper_scratch     82
Name: count, dtype: int64


In [3]:
encoder = LabelEncoder()
df["label_encoded"] = encoder.fit_transform(df["classes"])
print("Encoded classes:", encoder.classes_)
num_classes = len(encoder.classes_)
print("Number of classes:", num_classes)

Encoded classes: ['bumper_dent' 'bumper_scratch' 'door_dent' 'door_scratch' 'glass_shatter'
 'head_lamp' 'tail_lamp' 'unknown']
Number of classes: 8


In [4]:
IMG_SIZE = 224
images = []
labels = []

for _, row in df.iterrows():
    img = cv2.imread(row["image_path"])
    if img is None:
        continue
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = preprocess_input(img.astype(np.float32))
    images.append(img)
    labels.append(row["label_encoded"])

X = np.array(images, dtype=np.float32)
y = np.array(labels, dtype=np.int32)
print("Dataset shape:", X.shape, "Labels shape:", y.shape)

Dataset shape: (1512, 224, 224, 3) Labels shape: (1512,)


In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

Train: (1209, 224, 224, 3)
Validation: (151, 224, 224, 3)
Test: (152, 224, 224, 3)


In [6]:
datagen = ImageDataGenerator(
    rotation_range=25,
    zoom_range=0.25,
    shear_range=0.1,
    width_shift_range=0.12,
    height_shift_range=0.12,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="reflect"
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))
print("Class weights:", class_weights)

Class weights: {0: np.float64(1.4672330097087378), 1: np.float64(2.289772727272727), 2: np.float64(0.9877450980392157), 3: np.float64(1.228658536585366), 4: np.float64(1.3738636363636363), 5: np.float64(1.4257075471698113), 6: np.float64(1.386467889908257), 7: np.float64(0.34424829157175396)}


In [7]:
def build_vgg19_model(params, num_classes):
    base_model = VGG19(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    if params["unfreeze_last_n"] > 0:
        for layer in base_model.layers[:-params["unfreeze_last_n"]]:
            layer.trainable = False
        for layer in base_model.layers[-params["unfreeze_last_n"]:]:
            layer.trainable = True
    else:
        base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(params["dropout0"])(x)
    x = Dense(
        params["dense1"],
        activation="relu",
        kernel_regularizer=l2(params["l2_reg"])
    )(x)
    x = BatchNormalization()(x)
    x = Dropout(params["dropout1"])(x)
    x = Dense(
        params["dense2"],
        activation="relu",
        kernel_regularizer=l2(params["l2_reg"])
    )(x)
    x = BatchNormalization()(x)
    x = Dropout(params["dropout2"])(x)
    x = Dense(128, activation="relu")(x)
    x = BatchNormalization()(x)   # 🔥 ADDED
    x = Dropout(params["dropout3"])(x)
    output = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=base_model.input, outputs=output)
    optimizer = Adam(learning_rate=params["lr"], beta_1=params["momentum"])
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [11]:
class EHOAOptimizer:
    def __init__(
        self,
        pop_size=4,
        iterations=3,
        c1=1.5,
        c2=1.5,
        w=0.85,
        trial_epochs=3
    ):
        self.pop_size = pop_size
        self.iterations = iterations
        self.c1 = c1
        self.c2 = c2
        self.w = w
        self.trial_epochs = trial_epochs
        self.dim = 10
        self.dense1_choices = [256, 384, 512, 640]
        self.dense2_choices = [128, 192, 256, 320]
        self.unfreeze_choices = [0, 4, 8, 12, 16, 20]
        self.batch_choices = [16, 24, 32, 48]

    def decode(self, position):
        position = np.clip(position, 0.0, 1.0)
        lr = 10 ** np.interp(position[0], [0.0, 1.0], [np.log10(1e-5), np.log10(5e-4)])
        dropout0 = 0.20 + position[1] * 0.45
        dropout1 = 0.20 + position[2] * 0.45
        dropout2 = 0.15 + position[3] * 0.40
        dropout3 = 0.10 + position[4] * 0.35

        dense1_idx = min(int(position[5] * len(self.dense1_choices)), len(self.dense1_choices) - 1)
        dense2_idx = min(int(position[6] * len(self.dense2_choices)), len(self.dense2_choices) - 1)
        unfreeze_idx = min(int(position[7] * len(self.unfreeze_choices)), len(self.unfreeze_choices) - 1)
        batch_idx = min(int(position[8] * len(self.batch_choices)), len(self.batch_choices) - 1)

        dense1 = self.dense1_choices[dense1_idx]
        dense2 = self.dense2_choices[dense2_idx]
        unfreeze_last_n = self.unfreeze_choices[unfreeze_idx]
        batch_size = self.batch_choices[batch_idx]

        momentum = 0.82 + position[9] * 0.17
        l2_reg = 1e-6 + position[9] * (1e-3 - 1e-6)

        return {
            "lr": float(lr),
            "dropout0": float(dropout0),
            "dropout1": float(dropout1),
            "dropout2": float(dropout2),
            "dropout3": float(dropout3),
            "dense1": int(dense1),
            "dense2": int(dense2),
            "unfreeze_last_n": int(unfreeze_last_n),
            "batch_size": int(batch_size),
            "momentum": float(momentum),
            "l2_reg": float(l2_reg),
        }

    def fitness(self, params):
        K.clear_session()
        model = build_vgg19_model(params, num_classes)

        early_stop = EarlyStopping(
            monitor="val_accuracy",
            patience=2,
            restore_best_weights=True,
            verbose=0
        )
        reduce_lr = ReduceLROnPlateau(
            monitor="val_accuracy",
            factor=0.5,
            patience=1,
            min_lr=1e-6,
            verbose=0
        )

        history = model.fit(
            datagen.flow(X_train[:600], y_train_cat[:600], batch_size=params["batch_size"]),
            validation_data=(X_val, y_val_cat),
            epochs=self.trial_epochs,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        return float(np.max(history.history["val_accuracy"]))

    def optimize(self):
        positions = np.random.rand(self.pop_size, self.dim)
        velocities = np.zeros_like(positions)
        pbest = positions.copy()
        pbest_scores = np.full(self.pop_size, -np.inf)
        gbest = positions[0].copy()
        gbest_score = -np.inf
        best_params = None

        for iteration in range(self.iterations):
            alpha = 0.7 - 0.3 * (iteration / max(1, self.iterations - 1))
            beta = 1.0 - alpha
            scores = np.zeros(self.pop_size, dtype=np.float32)

            print(f"\nEHOA Search iteration {iteration + 1}/{self.iterations}")

            for i in range(self.pop_size):
                params = self.decode(positions[i])
                score = self.fitness(params)
                scores[i] = score

                print(f"Candidate {i + 1}: val_acc={score:.4f} | {params}")

                if score > pbest_scores[i]:
                    pbest_scores[i] = score
                    pbest[i] = positions[i].copy()

                if score > gbest_score:
                    gbest_score = score
                    gbest = positions[i].copy()
                    best_params = params.copy()

            if iteration < self.iterations - 1 and len(positions) > 3:
                keep_k = max(3, int(len(positions) * 0.75))
                keep_idx = np.argsort(scores)[-keep_k:]
                positions = positions[keep_idx]
                velocities = velocities[keep_idx]
                pbest = pbest[keep_idx]
                pbest_scores = pbest_scores[keep_idx]

            for i in range(len(positions)):
                r1 = np.random.rand(self.dim)
                r2 = np.random.rand(self.dim)
                velocities[i] = (
                    self.w * velocities[i]
                    + self.c1 * r1 * (pbest[i] - positions[i])
                    + self.c2 * r2 * (gbest - positions[i])
                )

                positions[i] = (
                    positions[i]
                    + alpha * (pbest[i] - positions[i])
                    + beta * (gbest - positions[i])
                    + velocities[i]
                )
                positions[i] = np.clip(positions[i], 0.0, 1.0)

            print(f"Best validation accuracy so far: {gbest_score:.4f}")

        return best_params, gbest_score

In [12]:
searcher = EHOAOptimizer(
    pop_size=3,
    iterations=2,
    c1=1.5,
    c2=1.5,
    w=0.85,
    trial_epochs=2
)

best_params, best_val_acc = searcher.optimize()
print("\nBest search parameters:")
print(best_params)
print(f"Best validation accuracy from search: {best_val_acc:.4f}")


EHOA Search iteration 1/2
Candidate 1: val_acc=0.3642 | {'lr': 0.00025817272189493276, 'dropout0': 0.5352455236775382, 'dropout1': 0.32441073327803527, 'dropout2': 0.1568060934758915, 'dropout3': 0.18306051563281917, 'dense1': 512, 'dense2': 128, 'unfreeze_last_n': 12, 'batch_size': 32, 'momentum': 0.8487969668011794, 'l2_reg': 0.00017022452843751918}
Candidate 2: val_acc=0.3642 | {'lr': 0.0001030686986715273, 'dropout0': 0.4731371425825552, 'dropout1': 0.41509016585932434, 'dropout2': 0.5312983230115497, 'dropout3': 0.13785711310447485, 'dense1': 384, 'dense2': 320, 'unfreeze_last_n': 8, 'batch_size': 24, 'momentum': 0.933606559674282, 'l2_reg': 0.0006686056065565158}
Candidate 3: val_acc=0.0993 | {'lr': 0.00013513645024312526, 'dropout0': 0.21030588129744446, 'dropout1': 0.4050638417508292, 'dropout2': 0.28340473787936377, 'dropout3': 0.3196933365013751, 'dense1': 512, 'dense2': 256, 'unfreeze_last_n': 12, 'batch_size': 24, 'momentum': 0.8618990099767614, 'l2_reg': 0.000247218299804

In [13]:
final_model = build_vgg19_model(best_params, num_classes)

loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
final_model.compile(
    optimizer=Adam(learning_rate=best_params["lr"], beta_1=best_params["momentum"]),
    loss=loss_fn,
    metrics=["accuracy"]
)

early_final = EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1)
reduce_final = ReduceLROnPlateau(monitor="val_loss", factor=0.4, patience=3, min_lr=1e-6, verbose=1)

history_stage1 = final_model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=best_params["batch_size"]),
    validation_data=(X_val, y_val_cat),
    epochs=30,
    class_weight=class_weights,
    callbacks=[early_final, reduce_final],
    verbose=1
)

for layer in final_model.layers[-24:]:
    layer.trainable = True

final_model.compile(
    optimizer=Adam(learning_rate=best_params["lr"] * 0.1, beta_1=best_params["momentum"]),
    loss=loss_fn,
    metrics=["accuracy"]
)

history_stage2 = final_model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=best_params["batch_size"]),
    validation_data=(X_val, y_val_cat),
    epochs=18,
    class_weight=class_weights,
    callbacks=[early_final, reduce_final],
    verbose=1
)

Epoch 1/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 250s 5s/step - accuracy: 0.1196 - loss: 2.8532 - val_accuracy: 0.1060 - val_loss: 2.7127 - learning_rate: 3.1191e-04
Epoch 2/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 252s 5s/step - accuracy: 0.1522 - loss: 2.6141 - val_accuracy: 0.3444 - val_loss: 2.0051 - learning_rate: 3.1191e-04
Epoch 3/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 253s 5s/step - accuracy: 0.1978 - loss: 2.5043 - val_accuracy: 0.3974 - val_loss: 2.3575 - learning_rate: 3.1191e-04
Epoch 4/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 252s 5s/step - accuracy: 0.2189 - loss: 2.4362 - val_accuracy: 0.3775 - val_loss: 2.0225 - learning_rate: 3.1191e-04
Epoch 5/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.1912 - loss: 2.4257
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0001247655716724694.
51/51 ━━━━━━━━━━━━━━━━━━━━ 252s 5s/step - accuracy: 0.1912 - loss: 2.4265 - val_accuracy: 0.3576 - val_loss: 5.1597 - learning_rate: 3.1191e-04
Epoch 6/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 251s 5s/step - accuracy: 0.2337 - loss:

KeyboardInterrupt: 

In [ ]:
def mc_dropout_predict(model, X, T=40, batch_size=32):
    predictions = []
    for _ in range(T):
        predictions.append(model(X, training=True).numpy())
    predictions = np.stack(predictions, axis=0)
    mean_pred = predictions.mean(axis=0)
    std_pred = predictions.std(axis=0)
    return mean_pred, std_pred

mean_pred, uncertainty = mc_dropout_predict(final_model, X_test, T=40)

pred_classes = np.argmax(mean_pred, axis=1)
confidence = np.max(mean_pred, axis=1)
uncertainty_score = np.mean(uncertainty, axis=1)

fraud_flags = []
severity = []
for c, u in zip(confidence, uncertainty_score):
    if c < 0.60 and u > 0.20:
        fraud_flags.append("HIGH RISK ⚠️")
    elif c < 0.75 or u > 0.18:
        fraud_flags.append("MEDIUM RISK ⚠️")
    else:
        fraud_flags.append("SAFE ✅")

    if c > 0.9 and u < 0.12:
        severity.append("LOW DAMAGE")
    elif c > 0.72:
        severity.append("MEDIUM DAMAGE")
    else:
        severity.append("HIGH DAMAGE")

In [ ]:
y_true = encoder.inverse_transform(y_test)
y_pred = encoder.inverse_transform(pred_classes)

print("Final evaluation on reserved test set")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average="weighted", zero_division=0))
print("Recall:", recall_score(y_true, y_pred, average="weighted", zero_division=0))
print("F1 Score:", f1_score(y_true, y_pred, average="weighted", zero_division=0))

print("\nClassification Report")
print(classification_report(y_true, y_pred, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - VGG19 + EHOA + MC Dropout")
plt.tight_layout()
plt.show()

risk_df = pd.DataFrame({
    "true_label": y_true,
    "predicted_label": y_pred,
    "confidence": confidence,
    "uncertainty": uncertainty_score,
    "risk_flag": fraud_flags,
    "severity": severity
})

print("\nHigh-risk samples:")
print(risk_df[risk_df["risk_flag"] == "HIGH RISK ⚠️"].head(10))

In [ ]:
final_model.save("vgg19_ehoa_novelty.h5")
print("Saved final model to vgg19_ehoa_novelty.h5")